# 1 · plot — Top7 contact map

Draws the dataset written by [`1_make_top7_heatmap_data.ipynb`](1_make_top7_heatmap_data.ipynb).
No GPU, no model: everything comes from the stored prediction.

Two figures, because the two are read differently — a single mirrored square when the reader
should see one object, and two panels when they should compare like with like:

* `top7_heatmap_mirrored` — prediction above the diagonal, observed contacts below.
* `top7_heatmap_side_by_side` — observed on the left, predicted on the right.

Both are masked to the pairs the metric scores: a blank region means *not scorable* (an
unresolved residue, or separation < 6), not *no contact*.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("notebooks/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
DATASET = "1_top7_heatmap"
DPI = 300                # PNG resolution; the PDF beside it is vector
metadata = figlib.describe(DATASET)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

figlib.figure_style(DPI)
directory = figlib.require(DATASET, "score.npy", "votes.npy", "ground_truth.json",
                           "ca_coords.npy")
score = np.load(directory / "score.npy")        # votes + tie-break; what the metric ranks
votes = np.load(directory / "votes.npy")        # the rollout count itself
n_rollouts = metadata["parameters"]["n_rollouts"]
truth_record = json.loads((directory / "ground_truth.json").read_text())

length = truth_record["L"]
truth = figlib.true_matrix(length, truth_record["contacts"])
resolved = np.zeros(length, bool)
resolved[np.asarray(truth_record["resolved"])] = True
scorable = np.outer(resolved, resolved)
scorable &= np.abs(np.subtract.outer(np.arange(length), np.arange(length))) >= figlib.MIN_SEPARATION
# With origin="lower", element [i, j] is drawn at x=j, y=i — so the VISUALLY upper-left
# triangle is i > j (np.tril), NOT np.triu. Getting this the wrong way round mirrors the
# figure while leaving the annotations reading correctly, which is as invisible as it sounds.
visually_upper = np.tril(np.ones((length, length), bool), k=-1)
# The colour is the fraction of rollouts that asserted the pair — an absolute 0-1 scale, not one
# normalised to this protein's strongest pair, so two proteins' maps are comparable and a washed
# out map means the model was never confident rather than that it had a weak best guess. The
# tie-break term is deliberately excluded: it orders tied pairs for the metric and is not
# evidence about a contact.
confidence = np.where(scorable, votes / n_rollouts, np.nan)
print(f"strongest pair: {np.nanmax(confidence):.0%} of {n_rollouts} rollouts · "
      f"{int(np.nansum(np.triu(confidence, k=1) > 0))} pairs got at least one vote")

# One colormap for "position along the chain", used by the ribbon and by the strips along the
# contact map's axes. Same colour = same residue in both figures, which is the whole point of
# drawing them together.
CHAIN_CMAP = "viridis"


def chain_strips(axis, length, cmap=CHAIN_CMAP, width=0.028, pad=0.012):
    """Paint an N-to-C colour bar along the bottom and left edges of a contact map.

    The ribbon figure colours the fold by the same scale, so a feature in the map can be found on
    the structure by matching colour instead of counting residues.
    """
    gradient = np.linspace(0, 1, length)[None, :]
    bottom = axis.inset_axes([0, -width - pad, 1, width])
    bottom.imshow(gradient, aspect="auto", cmap=cmap)
    left = axis.inset_axes([-width - pad, 0, width, 1])
    left.imshow(gradient.T, aspect="auto", cmap=cmap, origin="lower")
    for strip in (bottom, left):
        strip.set_xticks([]); strip.set_yticks([])
        for spine in strip.spines.values():
            spine.set_visible(False)
    return bottom, left


HEAT = LinearSegmentedColormap.from_list("votes", ["#FFFFFF", "#F4C36B", "#C44E52", "#3B0A0C"])
OBSERVED = LinearSegmentedColormap.from_list("gt", ["#2F2F2F", "#2F2F2F"])
PDB_ID = truth_record["stem"].split("_")[0].upper()
print(f"{truth_record['stem']}  ({PDB_ID})  L={length}  "
      f"{int(truth.sum() // 2)} observed contacts")

In [ ]:
# --- mirrored square ---------------------------------------------------------------------------
figure, axis = plt.subplots(figsize=(3.6, 3.6))
image = axis.imshow(np.where(visually_upper, confidence, np.nan), cmap=HEAT, origin="lower",
                    interpolation="none", vmin=0, vmax=1)
axis.imshow(np.where(~visually_upper & truth & scorable, 1.0, np.nan), cmap=OBSERVED, origin="lower",
            interpolation="none", vmin=0, vmax=1)
axis.plot([0, length - 1], [0, length - 1], color="0.75", lw=0.6)
axis.set(xlabel="residue", ylabel="residue")
axis.set_xticks([0, length // 2, length - 1])
axis.set_yticks([0, length // 2, length - 1])
for spine in ("top", "right"):
    axis.spines[spine].set_visible(True)
bar = figure.colorbar(image, ax=axis, fraction=0.046, pad=0.03, ticks=[0, 0.5, 1.0])
bar.set_label(f"fraction of {n_rollouts} rollouts asserting the contact", fontsize=8)
bar.ax.set_yticklabels(["0", "", "1"])
axis.text(0.03, 0.95, "Predicted", transform=axis.transAxes, ha="left", va="top", fontsize=8)
axis.text(0.97, 0.05, f"Observed ({PDB_ID})", transform=axis.transAxes, ha="right", va="bottom",
          fontsize=8)
chain_strips(axis, length)
figlib.save_figure(figure, "top7_heatmap_mirrored", DPI)
plt.show()

In [ ]:
# --- side by side ------------------------------------------------------------------------------
# constrained_layout with the colorbar attached to BOTH axes keeps the panels the same size;
# hanging it off the right panel alone silently shrinks that one.
figure, axes = plt.subplots(1, 2, figsize=(7.2, 3.5), sharey=True, layout="constrained")
axes[0].imshow(np.where(truth & scorable, 1.0, np.nan), cmap=OBSERVED, origin="lower",
               interpolation="none", vmin=0, vmax=1)
image = axes[1].imshow(confidence, cmap=HEAT, origin="lower", interpolation="none", vmin=0, vmax=1)
axes[0].set(xlabel="residue", ylabel="residue")
axes[1].set(xlabel="residue")
for axis, note in zip(axes, (f"Observed ({PDB_ID})", "Predicted")):
    axis.plot([0, length - 1], [0, length - 1], color="0.8", lw=0.6)
    axis.set_xticks([0, length // 2, length - 1])
    axis.set_yticks([0, length // 2, length - 1])
    axis.text(0.03, 0.95, note, transform=axis.transAxes, ha="left", va="top", fontsize=8)
    for spine in ("top", "right"):
        axis.spines[spine].set_visible(True)
bar = figure.colorbar(image, ax=list(axes), fraction=0.046, pad=0.02, ticks=[0, 0.5, 1.0])
bar.set_label(f"fraction of {n_rollouts} rollouts asserting the contact", fontsize=8)
bar.ax.set_yticklabels(["0", "", "1"])
for axis in axes:
    chain_strips(axis, length)
figlib.save_figure(figure, "top7_heatmap_side_by_side", DPI)
plt.show()

In [ ]:
# --- the fold itself, coloured N to C ----------------------------------------------------------
# A CA trace, smoothed, with each segment coloured by its position along the chain — the same
# scale painted along the contact map's axes above, so a contact in the map can be located on the
# structure by colour rather than by counting residues.
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy.interpolate import splev, splprep

ELEVATION = 18      # @param — viewing angle, degrees above the horizon
AZIMUTH = -60       # @param — viewing angle, degrees around
SMOOTHING = 800     # points along the interpolated backbone

ca_coords = np.load(directory / "ca_coords.npy")
present = ~np.isnan(ca_coords[:, 0])
trace = ca_coords[present]
print(f"{PDB_ID} chain {metadata['structure']['chain']}: {len(trace)} CA atoms, "
      f"residues {np.flatnonzero(present).min()}–{np.flatnonzero(present).max()}")

# A cubic spline through the CA atoms reads as a ribbon at print size; s=0 keeps it on the atoms.
spline, _ = splprep(trace.T, s=0, k=3)
along = np.linspace(0, 1, SMOOTHING)
curve = np.asarray(splev(along, spline)).T
segments = np.stack([curve[:-1], curve[1:]], axis=1)

figure = plt.figure(figsize=(4.0, 3.8))
axis = figure.add_subplot(projection="3d")
axis.add_collection3d(Line3DCollection(segments, colors=plt.get_cmap(CHAIN_CMAP)(along[:-1]),
                                       linewidth=2.6, capstyle="round"))
axis.scatter(*trace[0], color=plt.get_cmap(CHAIN_CMAP)(0.0), s=26, depthshade=False)
axis.scatter(*trace[-1], color=plt.get_cmap(CHAIN_CMAP)(1.0), s=26, depthshade=False)
axis.text(*trace[0], "  N", fontsize=9, weight="bold")
axis.text(*trace[-1], "  C", fontsize=9, weight="bold")

# Equal aspect, no box: this panel is a picture of a fold, not a plot of coordinates.
centre = trace.mean(axis=0)
radius = np.abs(trace - centre).max() * 1.05
axis.set_xlim(centre[0] - radius, centre[0] + radius)
axis.set_ylim(centre[1] - radius, centre[1] + radius)
axis.set_zlim(centre[2] - radius, centre[2] + radius)
axis.set_box_aspect((1, 1, 1))
axis.view_init(elev=ELEVATION, azim=AZIMUTH)
axis.set_axis_off()

mappable = plt.cm.ScalarMappable(cmap=CHAIN_CMAP)
mappable.set_array([0, 1])
bar = figure.colorbar(mappable, ax=axis, fraction=0.035, pad=0.02, ticks=[0, 1])
bar.set_label("position along the chain", fontsize=8)
bar.ax.set_yticklabels(["N", "C"])
figlib.save_figure(figure, "top7_structure_ribbon", DPI)
plt.show()

In [ ]:
# The first few statements of the real document, for a format panel.
print("".join((figlib.dataset_dir(DATASET) / "document.txt").read_text()[:220]), "...")